## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL")
model = ChatOpenAI(
    model="gpt-4.1",
    api_key = openai_api_key,
    base_url = openai_base_url
)
model

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001B5F10216A0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001B5F1022270>, root_client=<openai.OpenAI object at 0x000001B5F0AAE120>, root_async_client=<openai.AsyncOpenAI object at 0x000001B5F1021FD0>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://yibuapi.com/v1')

In [5]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What is the weather like in Boston?")
print(f"Response: {response}")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

Response: content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 52, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1', 'system_fingerprint': 'fp_e05894342c', 'id': 'chatcmpl-DK17vLm2C9ELL8mMo4mRMGU2KiP60', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019cf691-9afb-7892-95e8-d7508134eee5-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_EqNUCnrq8biqTmwHZuzuyg7g', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 52, 'output_tokens': 15, 'total_tokens': 67, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Tool: get_weather
Arguments: {'location': 'Boston'}


# Tool Execution Loops

In [9]:
messages = [{"role": "user", "content": "What is the weather like in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(f"Final Response: {final_response.text}")

Final Response: The weather in Boston is sunny.


In [10]:
messages

[{'role': 'user', 'content': 'What is the weather like in Boston?'},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 52, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1', 'system_fingerprint': 'fp_e05894342c', 'id': 'chatcmpl-DK1C5Z64G408vp3WUJqIonCFULvfK', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf695-8cec-7f71-84a5-b04321d06b17-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_NREnFL85TScBX7MAcXLNbwt7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 15, 'total_tokens': 67, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 